# 04 — Fine-tune 3 Model Deep Learning untuk Sentimen + Toksisitas

Loop atas 3 model (BERT-base, RoBERTa-base, DistilBERT) × 2 tugas (sentimen, toksisitas) = 6 fine-tuning runs.

Detoxify (`unitary/toxic-bert`) tidak di-fine-tune di sini — checkpoint pretrained dipakai langsung di notebook 05 inference (zero-shot reference untuk sentimen + specialist untuk toksisitas).

**Membutuhkan GPU CUDA**. Estimasi 4-12 jam tergantung hardware.

**Pra-syarat**:
- Dataset Jigsaw Toxic Comment Classification sudah di-download ke `data/external/jigsaw_toxic/{train,test,test_labels}.csv`.
- Dataset SST-2 / TweetEval-sentiment akan di-download otomatis via Hugging Face datasets.

**Output**:
- `models/<model>-sentiment/` dan `models/<model>-toxicity/` (checkpoint).
- `reports/training_<model>_<task>.log` (loss kurva, metrik, durasi).

In [2]:
!pip install datasets accelerate evaluate
%pip install datasets accelerate evaluate detoxify



  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached evaluate-0.4.6-py3-none-any.whl.metadata (9.5 kB)
Using cached accelerate-1.13.0-py3-none-any.whl (383 kB)
Using cached evaluate-0.4.6-py3-none-any.whl (84 kB)

   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ----------------------

ERROR: Could not install packages due to an OSError: [WinError 2] The system cannot find the file specified: 'C:\\Python310\\Scripts\\accelerate.exe' -> 'C:\\Python310\\Scripts\\accelerate.exe.deleteme'


[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached evaluate-0.4.6-py3-none-any.whl.metadata (9.5 kB)
Using cached evaluate-0.4.6-py3-none-any.whl (84 kB)

   ---------------------------------------- 0/2 [evaluate]
   ---------------------------------------- 0/2 [evaluate]
   ---------------------------------------- 0/2 [evaluate]
   ---------------------------------------- 0/2 [evaluate]
   ---------------------------------------- 0/2 [evaluate]

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [WinError 2] The system cannot find the file specified: 'c:\\Python310\\Scripts\\evaluate-cli.exe' -> 'c:\\Python310\\Scripts\\evaluate-cli.exe.deleteme'


[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Sel 1: Setup
import sys
from pathlib import Path
_here = Path.cwd()
_root = next((p for p in [_here, *_here.parents] if (p / 'src').is_dir() and (p / 'configs').is_dir()), _here)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
import os
os.chdir(_root)

from src.runtime import load_config, print_banner, RunLog
config = load_config('configs/experiment.yaml')
print_banner('04_training', config)
run_log = RunLog(notebook='04_training', config_path='configs/experiment.yaml')

import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
else:
    msg = 'CUDA tidak tersedia — training akan SANGAT lambat di CPU. Disarankan pakai GPU/Colab.'
    print(f'[WARN] {msg}')
    run_log.add_warning(msg)

SEED = int(config['seed'])

c:\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Notebook: 04_training
Experiment: thesis-sentiment-toxicity-dota2-decade
Seed: 42  |  Git: be03038
Started at: 2026-05-04T10:05:24+00:00Z
Versi paket:
  - python: 3.10.0
  - transformers: 5.5.0
  - torch: 2.8.0+cu128
  - datasets: 4.8.5
  - scikit-learn: 1.7.1
  - pandas: 2.3.3
  - numpy: 1.26.4
CUDA available: True
Device: NVIDIA GeForce RTX 3060


In [7]:
# Sel baru di atas sel 4
import importlib
import src.training.sentiment_trainer
import src.training.toxicity_trainer
importlib.reload(src.training.sentiment_trainer)
importlib.reload(src.training.toxicity_trainer)
from src.training.sentiment_trainer import fine_tune_sentiment
from src.training.toxicity_trainer import fine_tune_toxicity


In [8]:
# Sel 2: Konfigurasi 3 model yang akan di-fine-tune
MODELS = {
    'bert': config['model_checkpoints']['bert']['hf_repo'],
    'roberta': config['model_checkpoints']['roberta']['hf_repo'],
    'distilbert': config['model_checkpoints']['distilbert']['hf_repo'],
}
TASKS = ['sentiment', 'toxicity']
HP_SENT = config['hyperparameters']['sentiment']
HP_TOX = config['hyperparameters']['toxicity']
MODELS_ROOT = Path('models')
MODELS_ROOT.mkdir(exist_ok=True)

for name, repo in MODELS.items():
    print(f'  {name:12s} → {repo}')

  bert         → bert-base-uncased
  roberta      → roberta-base
  distilbert   → distilbert-base-uncased


In [9]:
# Sel 3: Load dataset publik (sentimen + toksisitas)
from src.training.dataset_loaders import load_sentiment_dataset, load_toxicity_dataset

# Pilihan sumber sentimen — TweetEval lebih dekat ke domain chat dibanding SST-2
# (movie reviews). Lihat configs/experiment.yaml#public_datasets.sentiment.
SENT_SOURCE = config['public_datasets']['sentiment'].get('name', 'sst2')
TOX_SOURCE = 'jigsaw'

print(f'Sentiment source: {SENT_SOURCE}')
sent_split = load_sentiment_dataset(source=SENT_SOURCE, cache_dir=Path('data/external'))
print(f'  train={len(sent_split.train):,}  val={len(sent_split.validation):,}  test={len(sent_split.test):,}')
print(f'  labels={sent_split.label_names}')

print(f'\nToxicity source: {TOX_SOURCE}')
try:
    tox_split = load_toxicity_dataset(source=TOX_SOURCE, cache_dir=Path('data/external'))
    print(f'  train={len(tox_split.train):,}  val={len(tox_split.validation):,}  test={len(tox_split.test):,}')
    print(f'  labels={tox_split.label_names}')
except FileNotFoundError as e:
    print(f'[ERROR] {e}')
    print('Download Jigsaw dari Kaggle dulu, lalu re-run sel ini.')
    tox_split = None

Sentiment source: tweeteval-sentiment
  train=45,615  val=2,000  test=12,284
  labels=['negative', 'neutral', 'positive']

Toxicity source: jigsaw
  train=143,614  val=15,957  test=63,978
  labels=['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']


In [10]:
# Sel 4: Loop fine-tune 3 model × 2 tugas
from src.training.sentiment_trainer import fine_tune_sentiment
from src.training.toxicity_trainer import fine_tune_toxicity
import json
import time

# Threshold validasi sesuai spec sentiment-toxicity-modeling
SENT_F1_MACRO_THRESHOLD = 0.70
TOX_F1_MICRO_THRESHOLD = 0.85

results = {}

for model_key, model_repo in MODELS.items():
    # Sentimen
    out_dir = MODELS_ROOT / f'{model_key}-sentiment'
    print(f'\n=== Fine-tune {model_key} → SENTIMEN → {out_dir} ===')
    t0 = time.time()
    res = fine_tune_sentiment(
        model_name=model_repo,
        train_df=sent_split.train,
        val_df=sent_split.validation,
        label_names=sent_split.label_names,
        output_dir=out_dir,
        hyperparameters=HP_SENT,
        seed=SEED,
    )
    elapsed = time.time() - t0
    print(f'  done in {elapsed:.0f}s  |  metrics={res.eval_metrics}')
    f1m = res.eval_metrics.get('eval_f1_macro', 0)
    if f1m < SENT_F1_MACRO_THRESHOLD:
        msg = f'{model_key}/sentiment validation F1-macro={f1m:.3f} < threshold {SENT_F1_MACRO_THRESHOLD} (spec)'
        print(f'  [WARN] {msg}')
        run_log.add_warning(msg)
    results[(model_key, 'sentiment')] = res
    log_path = Path('reports') / f'training_{model_key}_sentiment.log'
    log_path.write_text(json.dumps({'metrics': res.eval_metrics, 'history': res.history, 'duration_sec': elapsed}, indent=2, default=str))
    run_log.add_output(out_dir)
    run_log.add_output(log_path)

    # Toksisitas
    if tox_split is None:
        msg = f'Skip toxicity fine-tune untuk {model_key} — dataset Jigsaw tidak tersedia.'
        print(f'[WARN] {msg}')
        run_log.add_warning(msg)
        continue
    out_dir = MODELS_ROOT / f'{model_key}-toxicity'
    print(f'\n=== Fine-tune {model_key} → TOKSISITAS → {out_dir} ===')
    t0 = time.time()
    res = fine_tune_toxicity(
        model_name=model_repo,
        train_df=tox_split.train,
        val_df=tox_split.validation,
        label_names=tox_split.label_names,
        output_dir=out_dir,
        hyperparameters=HP_TOX,
        seed=SEED,
    )
    elapsed = time.time() - t0
    print(f'  done in {elapsed:.0f}s  |  metrics={res.eval_metrics}')
    f1mi = res.eval_metrics.get('eval_f1_micro', 0)
    if f1mi < TOX_F1_MICRO_THRESHOLD:
        msg = f'{model_key}/toxicity validation F1-micro={f1mi:.3f} < threshold {TOX_F1_MICRO_THRESHOLD} (spec)'
        print(f'  [WARN] {msg}')
        run_log.add_warning(msg)
    results[(model_key, 'toxicity')] = res
    log_path = Path('reports') / f'training_{model_key}_toxicity.log'
    log_path.write_text(json.dumps({'metrics': res.eval_metrics, 'history': res.history, 'duration_sec': elapsed}, indent=2, default=str))
    run_log.add_output(out_dir)
    run_log.add_output(log_path)


=== Fine-tune bert → SENTIMEN → models\bert-sentiment ===


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2724.14it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider tr

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,Precision Macro,Recall Macro
1,0.593079,0.612147,0.725500,0.707942,0.727008,0.701647
2,0.525371,0.606809,0.752000,0.727592,0.746386,0.715036
3,0.368311,0.691083,0.744000,0.726775,0.724429,0.729512


Writing model shards: 100%|██████████| 1/1 [00:18<00:00, 18.15s/it]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer

  done in 2430s  |  metrics={'eval_loss': 0.6910828948020935, 'eval_accuracy': 0.744, 'eval_f1_macro': 0.7267750669769839, 'eval_precision_macro': 0.7244290864797777, 'eval_recall_macro': 0.7295124589428387, 'eval_runtime': 4.4012, 'eval_samples_per_second': 454.42, 'eval_steps_per_second': 14.314}

=== Fine-tune bert → TOKSISITAS → models\bert-toxicity ===


...

  done in 456s  |  metrics={'eval_loss': 0.6581937074661255, 'eval_accuracy': 0.739, 'eval_f1_macro': 0.7236417818799193, 'eval_precision_macro': 0.7202564128961532, 'eval_recall_macro': 0.7275202528367085, 'eval_runtime': 0.9906, 'eval_samples_per_second': 2018.885, 'eval_steps_per_second': 63.595}

=== Fine-tune distilbert → TOKSISITAS → models\distilbert-toxicity ===


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 10063.35it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Map: 100%|██████████| 15957/15957 [00:00<00:00, 16028.50 examples/s]
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,Hamming Loss,Avg Precision Macro,Runtime,Samples Per Second,Steps Per Second
1,0.037684,0.038349,0.788453,0.538804,0.015385,0.682265,22.266800,716.628000,22.410000
2,0.028009,0.036160,0.794770,0.646474,0.015082,0.711812,21.160900,754.080000,23.581000
3,0.024863,0.036950,0.795915,0.646407,0.014612,0.707567,21.146800,754.582000,23.597000


Writing model shards: 100%|██████████| 1/1 [00:05<00:00,  5.60s/it]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].
Writing model shards: 100%|██████████| 1/1 [00:07<00:00,  7.21s/it]

  done in 2216s  |  metrics={'eval_loss': 0.036950379610061646, 'eval_f1_micro': 0.7959153902261124, 'eval_f1_macro': 0.6464072310235878, 'eval_hamming_loss': 0.01461218691901151, 'eval_avg_precision_macro': 0.7075666112048191, 'eval_runtime': 21.1468, 'eval_samples_per_second': 754.582, 'eval_steps_per_second': 23.597}
  [WARN] distilbert/toxicity validation F1-micro=0.796 < threshold 0.85 (spec)


In [11]:
# Sel 5: Pin revision SHA Hugging Face di manifest + tulis models/detoxify/manifest.json
import json
import yaml
from huggingface_hub import HfApi
api = HfApi()

cfg_path = Path('configs/experiment.yaml')
cfg = yaml.safe_load(cfg_path.read_text(encoding='utf-8'))
for model_key, repo in MODELS.items():
    try:
        info = api.model_info(repo)
        cfg['model_checkpoints'][model_key]['revision_sha'] = info.sha
        print(f'  {model_key}: {repo} @ {info.sha[:12]}')
    except Exception as e:
        msg = f'Gagal ambil revision SHA untuk {repo}: {e}'
        print(f'[WARN] {msg}')
        run_log.add_warning(msg)

# Detoxify: tidak di-fine-tune di sini, tapi spec mensyaratkan models/detoxify/manifest.json
detox_repo = cfg['model_checkpoints']['detoxify']['hf_repo']
detox_sha = ''
try:
    info = api.model_info(detox_repo)
    detox_sha = info.sha
    cfg['model_checkpoints']['detoxify']['revision_sha'] = info.sha
    print(f'  detoxify: @ {info.sha[:12]}')
except Exception as e:
    print(f'[WARN] {e}')

detox_dir = MODELS_ROOT / 'detoxify'
detox_dir.mkdir(parents=True, exist_ok=True)
detox_manifest = {
    'hf_repo': detox_repo,
    'revision_sha': detox_sha,
    'note': 'pretrained, not fine-tuned — dipakai langsung di notebook 05 untuk toksisitas (specialist) dan zero-shot mapping ke sentimen di notebook 06.',
    'task': 'toxicity_multi_label',
    'labels': cfg['labels']['toxicity_labels'],
}
(detox_dir / 'manifest.json').write_text(json.dumps(detox_manifest, indent=2), encoding='utf-8')
print(f'  detoxify manifest tertulis: {detox_dir / "manifest.json"}')
run_log.add_output(detox_dir / 'manifest.json')

cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')
run_log.add_output(cfg_path)

  bert: bert-base-uncased @ 86b5e0934494
  roberta: roberta-base @ e2da8e2f811d
  distilbert: distilbert-base-uncased @ 12040accade4
  detoxify: @ 4d6c22e74ba2
  detoxify manifest tertulis: models\detoxify\manifest.json


In [12]:
run_log.save('reports/run_log.csv')

[run_log] 04_training → 24965.52s, 14 outputs, 3 warnings → reports\run_log.csv
